# RAG PDF Question Answering Project

This notebook builds a simple **RAG-based Question Answering system** using:

- LangChain
- PDF document loader
- Hugging Face BGE embeddings
- FAISS vector database
- Hugging Face LLM

## Objective

The objective is to load a PDF document, split it into chunks, convert those chunks into embeddings, store them in FAISS, retrieve the most relevant chunks for a user question, and generate an answer using an LLM.

## Sample PDF Used

**Health Insurance Coverage Status and Type by Geography: 2021 and 2022**  
Source: U.S. Census Bureau  
File name used in this project: `acsbr-015-health-insurance-coverage.pdf`


## 1. Install Required Libraries

Run this cell once if the packages are not installed in your environment.

In [ ]:
# Install required packages if needed
# !pip install langchain langchain-community sentence-transformers faiss-cpu pypdf huggingface_hub

## 2. Import Libraries

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.llms import HuggingFaceHub

import os
from pathlib import Path

## 3. Load the PDF Document

Place the PDF file in the same folder as this notebook, or update the path below.

In [ ]:
pdf_path = "acsbr-015-health-insurance-coverage.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Total PDF pages loaded: {len(documents)}")
print(documents[0].page_content[:500])

## 4. Split the PDF into Chunks

The PDF content is split into smaller chunks because LLMs cannot efficiently process very large documents at once.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

final_documents = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(final_documents)}")
print(final_documents[0].page_content[:500])

## 5. Create Hugging Face Embeddings

Embeddings convert text into numerical vectors so that FAISS can search based on meaning.

In [ ]:
huggingface_embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

sample_vector = huggingface_embeddings.embed_query(final_documents[0].page_content)
print(f"Embedding vector length: {len(sample_vector)}")

## 6. Store Chunks in FAISS Vector Database

FAISS stores document vectors and helps retrieve the most relevant chunks for a question.

In [ ]:
vectorstore = FAISS.from_documents(final_documents, huggingface_embeddings)
print("FAISS vector store created successfully.")

## 7. Test Similarity Search

This checks whether FAISS can find the correct document section for a question.

In [ ]:
query = "What is health insurance coverage?"
relevant_documents = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(relevant_documents, start=1):
    print(f"
--- Relevant Chunk {i} ---")
    print(doc.page_content[:700])

## 8. Create Retriever

The retriever fetches the top matching chunks for each user question.

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(retriever)

## 9. Configure Hugging Face LLM

You need a Hugging Face API token to use the hosted model.

Recommended model for better answers:

`mistralai/Mistral-7B-Instruct-v0.2`

Do not hardcode your token in production. Use environment variables.

In [ ]:
# Add your Hugging Face token here or set it in your system environment
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your_token_here"

hf = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    model_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 500
    }
)

## 10. Create RAG Prompt Template

The prompt instructs the LLM to answer only from the retrieved context.

In [ ]:
prompt_template = """
You are a helpful assistant answering questions from a PDF document.
Use only the given context to answer the question.
If the answer is not available in the context, say: "The document does not mention this information."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

## 11. Create RetrievalQA Chain

This combines the retriever, prompt, and LLM.

In [ ]:
retrievalQA = RetrievalQA.from_chain_type(
    llm=hf,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

## 12. Ask Questions from the PDF

In [ ]:
query = "What is health insurance coverage?"

result = retrievalQA.invoke({"query": query})

print("Answer:")
print(result["result"])

print("
Source Documents:")
for i, source_doc in enumerate(result["source_documents"], start=1):
    print(f"
Source {i}: Page {source_doc.metadata.get('page')}")
    print(source_doc.page_content[:500])

## 13. More Sample Questions

Try these questions:

1. What is health insurance coverage?
2. What was the uninsured rate in 2022?
3. Which state had the lowest uninsured rate in 2022?
4. Which state had the highest uninsured rate in 2022?
5. What is the difference between private and public health insurance?
6. What changed in uninsured rates from 2021 to 2022?

In [ ]:
sample_questions = [
    "What is health insurance coverage?",
    "What was the uninsured rate in 2022?",
    "Which state had the lowest uninsured rate in 2022?",
    "Which state had the highest uninsured rate in 2022?",
    "What is the difference between private and public health insurance?",
    "What changed in uninsured rates from 2021 to 2022?"
]

for question in sample_questions:
    print("
Question:", question)
    response = retrievalQA.invoke({"query": question})
    print("Answer:", response["result"])
    print("-" * 80)

## Project Summary

This notebook demonstrates a complete beginner-level RAG pipeline:

1. Load PDF
2. Split text into chunks
3. Create embeddings
4. Store vectors in FAISS
5. Retrieve relevant chunks
6. Send context to LLM
7. Generate answer from document content

You can replace the sample PDF with your own BRD, SOP, user manual, API document, or POC report and use the same logic.